# Mix segmentation

In [1]:
import os
import json
import yaml
import numpy as np
import librosa
import pandas as pd
import matplotlib.pyplot as plt
import IPython
import soundfile as sf
from tqdm import tqdm 
from collections import OrderedDict
from sklearn.model_selection import train_test_split
import glob

## Setup

In [2]:
BASE_DIR = "."  


MEDLEYDB_ROOT = os.path.join(BASE_DIR, "MedleyDB")
AUDIO_DIR = os.path.join(MEDLEYDB_ROOT, "Audio")
ANNOTATIONS_DIR = os.path.join(MEDLEYDB_ROOT, "Annotations")


INSTRUMENT_ACTIVATIONS_DIR = os.path.join(ANNOTATIONS_DIR, "Instrument_Activations")
ACTIVATION_CONF_DIR = os.path.join(INSTRUMENT_ACTIVATIONS_DIR, "ACTIVATION_CONF")
SOURCEID_DIR = os.path.join(INSTRUMENT_ACTIVATIONS_DIR, "SOURCEID")

Define the instrument mapping from all the dataset categories into 18 custom classes:

In [ ]:
# First, we define the mapping we already agreed on
# Mapping from original instruments to grouped categories
instrument_mapping = {
    # Large individual classes
    'electric_bass': 'electric_bass',
    'drum_set': 'drum_set',
    'piano': 'piano',
    'tack_piano': 'piano',
    'synthesizer': 'synthesizer',
    'electric_piano': 'synthesizer',
    'clean_electric_guitar': 'clean_electric_guitar',
    'distorted_electric_guitar': 'distorted_electric_guitar',
    'acoustic_guitar': 'acoustic_guitar',
    'violin': 'violin',
    'female_singer': 'female_singer',
    'vocalists': 'vocalists',
    
    # Male vocals
    'male_singer': 'male_vocals',
    'male_speaker': 'male_vocals',
    'male_rapper': 'male_vocals',
    
    # Other bowed strings
    'viola': 'other_bowed_strings',
    'cello': 'other_bowed_strings',
    'double_bass': 'other_bowed_strings',
    'violin_section': 'other_bowed_strings',
    'viola_section': 'other_bowed_strings',
    'cello_section': 'other_bowed_strings',
    'string_section': 'other_bowed_strings',
    'erhu': 'other_bowed_strings',
    
    # Woodwinds
    'flute': 'woodwinds',
    'clarinet': 'woodwinds',
    'oboe': 'woodwinds',
    'bassoon': 'woodwinds',
    'tenor_saxophone': 'woodwinds',
    'soprano_saxophone': 'woodwinds',
    'alto_saxophone': 'woodwinds',
    'baritone_saxophone': 'woodwinds',
    'bass_clarinet': 'woodwinds',
    'piccolo': 'woodwinds',
    'flute_section': 'woodwinds',
    'clarinet_section': 'woodwinds',
    
    # Brass
    'trumpet': 'brass',
    'trombone': 'brass',
    'french_horn': 'brass',
    'french_horn_section': 'brass',
    'brass_section': 'brass',
    'tuba': 'brass',
    'horn_section': 'brass',
    'trumpet_section': 'brass',
    
    # Other percussion
    'auxiliary_percussion': 'other_percussion',
    'tambourine': 'other_percussion',
    'cymbals': 'other_percussion',
    'timpani': 'other_percussion',
    'tabla': 'other_percussion',
    'shaker': 'other_percussion',
    'snare_drum': 'other_percussion',
    'bass_drum': 'other_percussion',
    'kick_drum': 'other_percussion',
    'bongo': 'other_percussion',
    'darbuka': 'other_percussion',
    'doumbek': 'other_percussion',
    'gong': 'other_percussion',
    'toms': 'other_percussion',
    'drum_machine': 'other_percussion',
    'claps': 'other_percussion',
    'chimes': 'other_percussion',
    
    # Other plucked strings
    'mandolin': 'other_plucked_strings',
    'banjo': 'other_plucked_strings',
    'guzheng': 'other_plucked_strings',
    'oud': 'other_plucked_strings',
    'harp': 'other_plucked_strings',
    'lap_steel_guitar': 'other_plucked_strings',
    'yangqin': 'other_plucked_strings',
    'zhongruan': 'other_plucked_strings',
    'glockenspiel': 'other_plucked_strings',
    'vibraphone': 'other_plucked_strings',
    
    # Reed instruments
    'accordion': 'reed_instruments',
    'harmonica': 'reed_instruments',
    'melodica': 'reed_instruments',
    
    # Effects and processed sounds
    'fx_slash_processed_sound': 'fx_and_processed',
    'scratches': 'fx_and_processed'
}

# List of instruments to completely discard
instruments_to_discard = [
    'sampler',
    'main_system',
    'trombone_section',
]

# Final categories
instrument_categories = [
    'electric_bass',
    'drum_set',
    'piano',
    'synthesizer',
    'clean_electric_guitar',
    'distorted_electric_guitar',
    'acoustic_guitar',
    'violin',
    'female_singer',
    'vocalists',
    'male_vocals',
    'other_bowed_strings',
    'woodwinds',
    'brass',
    'other_percussion',
    'other_plucked_strings',
    'reed_instruments',
    'fx_and_processed'
]

# Mapping from categories to indices
category_to_idx = {category: idx for idx, category in enumerate(instrument_categories)}

Utility functions:

In [ ]:
def get_metadata_file_path(track_name):
    """Gets the path to the YAML metadata file for a track"""

    return os.path.join(MEDLEYDB_ROOT, "Audio", track_name, f"{track_name}_METADATA.yaml")


def plot_instrument_histogram(instrument_counts, top_n=20, min_count=None):
    """
    Creates a horizontal histogram with instrument frequencies.
    
    Args:
        instrument_counts: Dictionary with instrument as key and frequency as value
        top_n: Show only the top N most frequent instruments (default 20)
        min_count: Minimum appearance threshold to include an instrument
    """

    # Sort instruments by frequency (descending)
    sorted_instruments = sorted(instrument_counts.items(), key=lambda x: x[1], reverse=True)
    
    # Filter according to parameters
    filtered_instruments = []
    for instrument, count in sorted_instruments:
        if min_count is not None and count < min_count:
            continue
        filtered_instruments.append((instrument, count))
        if len(filtered_instruments) >= top_n:
            break
    
    # Separate names and counts for the plot
    names = [item[0] for item in filtered_instruments]
    counts = [item[1] for item in filtered_instruments]
    
    # Create figure with size adapted to the number of instruments
    plt.figure(figsize=(10, max(8, len(names) * 0.3)))
    
    # Create horizontal bars
    plt.barh(names, counts, color='skyblue')
    
    # Add labels and title
    plt.xlabel('Number of appearances')
    plt.title('Instrument frequency in MedleyDB')
    
    # Adjust margins
    plt.tight_layout()
    
    # Add the numeric value at the end of each bar
    for i, count in enumerate(counts):
        plt.text(count + 0.5, i, str(count), va='center')
    
    # Show the plot
    plt.show()

def get_activation_intervals(track_name, instrument):
    """
    Gets activation intervals for a specific instrument.
    
    Args:
        track_name: Track name
        instrument: Instrument name
        
    Returns:
        list: List of tuples (start_time, end_time)
    """

    sourceid_path = os.path.join(SOURCEID_DIR, f"{track_name}_SOURCEID.lab")
    
    intervals = []
    if os.path.exists(sourceid_path):
        with open(sourceid_path, 'r') as f:
            header = f.readline()  # Skip header
            for line in f:
                parts = line.strip().split(',')
                if len(parts) >= 3 and parts[2].strip() == instrument:
                    start_time = float(parts[0])
                    end_time = float(parts[1])
                    intervals.append((start_time, end_time))
    
    return intervals

def sanitize_directory_name(name):
    """
    Converts an instrument name into a safe directory name.
    
    Args:
        name: Original instrument name
        
    Returns:
        str: Sanitized name safe to use as a directory name
    """

    replacements = {
        '/': '_slash_',
        '\\': '_backslash_',
        ':': '_colon_',
        '*': '_star_',
        '?': '_question_',
        '"': '_quote_',
        '<': '_lt_',
        '>': '_gt_',
        '|': '_pipe_',
        ' ': '_'
    }
    
    result = name
    for char, replacement in replacements.items():
        result = result.replace(char, replacement)
    
    return result


def load_audio_segment(filepath, start_time, duration, sr=44100):
    """
    Loads an audio segment given an offset and duration.
    
    Args:
        filepath: Path to the .wav file
        start_time: Segment start in seconds
        duration: Segment duration in seconds
        sr: Sample rate
        
    Returns:
        np.ndarray: Audio array
    """
    
    try:
        y, _ = librosa.load(filepath, sr=sr, offset=start_time, duration=duration)
        return y
    except Exception as e:
        print(f"Error loading segment from {filepath}: {e}")
        return None


def get_mix_path(track_name):
    """
    Returns the path to the mix (MIX) file for a track.
    
    Args:
        track_name: Track name
        
    Returns:
        str: Full path to the MIX (.wav) file
    """
    return os.path.join(AUDIO_DIR, track_name, f"{track_name}_MIX.wav")


def get_constant_label_intervals(track_name, instrument_list, segment_duration=3.0):
    """
    Finds intervals where the set of active instruments remains constant.
    
    Args:
        track_name: Track name
        instrument_list: List of instruments in STEM order
        segment_duration: Minimum required duration to accept an interval
    
    Returns:
        list: List of tuples (start_time, end_time, active_instruments)
    """

    sourceid_path = os.path.join(SOURCEID_DIR, f"{track_name}_SOURCEID.lab")
    if not os.path.exists(sourceid_path):
        print(f"SOURCEID not found for {track_name}")
        return []

    # Step 1: Load all events
    events = []
    with open(sourceid_path, 'r') as f:
        header = f.readline()
        for line in f:
            parts = line.strip().split(',')
            if len(parts) >= 3:
                start = float(parts[0])
                end = float(parts[1])
                instrument = parts[2].strip()
                if instrument in instrument_list:
                    events.append((start, 'on', instrument))
                    events.append((end, 'off', instrument))
    
    # Step 2: Sort events by time
    events.sort()

    # Step 3: Iterate through events and detect changes
    intervals = []
    current_time = 0.0
    active_instruments = set()
    
    for event_time, event_type, instrument in events:
        if event_time > current_time and len(active_instruments) > 0:
            duration = event_time - current_time
            if duration >= segment_duration:
                intervals.append((current_time, event_time, set(active_instruments)))
        # Update current state
        if event_type == 'on':
            active_instruments.add(instrument)
        elif event_type == 'off':
            active_instruments.discard(instrument)
        current_time = event_time
    
    return intervals

def get_constant_category_intervals(track_name, segment_duration=3.0):
    """
    Finds intervals where the set of active categories remains constant.

    Args:
        track_name: Track name
        segment_duration: Minimum required duration to accept an interval

    Returns:
        list: List of tuples (start_time, end_time, active_categories)
    """

    sourceid_path = os.path.join(SOURCEID_DIR, f"{track_name}_SOURCEID.lab")
    if not os.path.exists(sourceid_path):
        print(f"SOURCEID not found for {track_name}")
        return []

    # 1. Read events and map to categories
    raw_events = []
    with open(sourceid_path, 'r') as f:
        f.readline()  # skip header
        for line in f:
            parts = line.strip().split(',')
            if len(parts) >= 3:
                start = float(parts[0])
                end = float(parts[1])
                instr = sanitize_directory_name(parts[2].strip())
                if instr in instrument_mapping:
                    category = instrument_mapping[instr]
                    raw_events.append((start, 'on', category))
                    raw_events.append((end, 'off', category))
    
    if not raw_events:
        return []

    # 2. Sort events chronologically
    raw_events.sort()

    # 3. Iterate through events to form intervals
    current_time = 0.0
    active_categories = set()
    intervals = []

    for event_time, event_type, category in raw_events:
        if event_time > current_time and len(active_categories) > 0:
            duration = event_time - current_time
            if duration >= segment_duration:
                intervals.append((current_time, event_time, set(active_categories)))
        if event_type == 'on':
            active_categories.add(category)
        elif event_type == 'off':
            active_categories.discard(category)
        current_time = event_time

    return intervals


def get_stem_instruments(track_name):
    """
    Returns a list of instruments in STEM order for a given track.

    Args:
        track_name: Track name

    Returns:
        dict: Dictionary {stem_id: instrument} ordered by key
    """
    
    metadata_path = get_metadata_file_path(track_name)

    if not os.path.exists(metadata_path):
        print(f"  ⚠️  Metadata not found: {metadata_path}")
        return {}

    with open(metadata_path, 'r') as f:
        metadata = yaml.safe_load(f)

    stem_instruments = {}
    if 'stems' in metadata:
        for stem_id in sorted(metadata['stems'].keys()):  # Sort by S01, S02...
            instr = metadata['stems'][stem_id].get('instrument')
            if instr:
                stem_instruments[stem_id] = instr

    return stem_instruments

WE NEED TO REMOVE THIS??????

In [ ]:
def extract_multilabel_segments_old(track_names, segment_duration=3.0, sr=44100, max_segments_per_track=None):
    """
    Extracts mix segments with multilabel tags where the active instruments remain constant.

    Args:
        track_names: List of track names to process
        segment_duration: Desired duration of each segment (in seconds)
        sr: Sample rate for loading audio
        max_segments_per_track: Maximum number of segments to extract per track (None = no limit)
    """
    
    for track_name in track_names:
        print(f"\nProcessing track: {track_name}")
        mix_path = get_mix_path(track_name)

        # Get all instruments present according to activation files
        stem_instruments = get_stem_instruments(track_name)
        instrument_list = list(stem_instruments.values())

        # Get intervals with constant labels
        intervals = get_constant_category_intervals(track_name, segment_duration=segment_duration)
        print(f"  → {len(intervals)} intervals of interest found")
        for i, (start_time, end_time, active_instruments) in enumerate(intervals):
            print(f"    Interval {i + 1}: {start_time:.2f}s – {end_time:.2f}s  | Active: {sorted(active_instruments)}")

        segment_count = 0
        for start_time, end_time, active_instruments in intervals:
            if max_segments_per_track and segment_count >= max_segments_per_track:
                break

            # Convert to grouped categories
            categories = set(active_instruments)

            if not categories:
                continue  # Skip if there are no valid classes

            # Multi-hot vector
            label_vector = np.zeros(len(category_to_idx), dtype=np.float32)
            for cat in categories:
                idx = category_to_idx[cat]
                label_vector[idx] = 1.0

            # Extract audio segment
            audio = load_audio_segment(mix_path, start_time, segment_duration, sr=sr)
            if audio is None:
                continue

            # Useful information
            print(f"    Segment {segment_count + 1}:")
            print(f"      Time: {start_time:.2f}s – {start_time + segment_duration:.2f}s")
            print(f"      Categories: {sorted(categories)}")
            print(f"      Multi-hot labels: {label_vector.tolist()}")
            print(f"      Audio duration: {len(audio)/sr:.2f} seconds")

            segment_count += 1


In [ ]:
def extract_multilabel_segments(track_names, segment_duration=3.0, sr=44100, max_segments_per_track=None):
    """
    Extracts mix segments with multilabel tags where the active instruments remain constant,
    and saves the results in individual folders as well as global lists.
    """
    
    os.makedirs("Mix_features", exist_ok=True)
    all_mels, all_cqts, all_labels = [], [], []

    for track_name in track_names:
        print(f"\nProcessing track: {track_name}")
        mix_path = get_mix_path(track_name)

        stem_instruments = get_stem_instruments(track_name)
        instrument_list = list(stem_instruments.values())

        intervals = get_constant_category_intervals(track_name, segment_duration=segment_duration)
        print(f"  → {len(intervals)} intervals of interest found for {track_name}")
        for i, (start_time, end_time, active_instruments) in enumerate(intervals):
            print(f"    Interval {i + 1}: {start_time:.2f}s – {end_time:.2f}s  | Active: {sorted(active_instruments)}")

        segment_count = 0
        all_segments = []

        # Calculate all possible segments in all intervals
        for start_time, end_time, active_instruments in intervals:
            hop = segment_duration / 2  # → 1.5s for 50% overlap
            num_possible_segments = int((end_time - start_time - segment_duration) // hop) + 1
            for i in range(num_possible_segments):
                seg_start = start_time + i * hop
                all_segments.append((seg_start, active_instruments))

        for seg_start_time, active_instruments in tqdm(all_segments, desc=f"Extracting segments from {track_name}"):
            if max_segments_per_track and segment_count >= max_segments_per_track:
                break

            categories = set(active_instruments)
            if not categories:
                continue

            label_vector = np.zeros(len(category_to_idx), dtype=np.float32)
            for cat in categories:
                label_vector[category_to_idx[cat]] = 1.0
            print(label_vector)
            audio = load_audio_segment(mix_path, seg_start_time, segment_duration, sr=sr)
            if audio is None:
                continue

            # Compute features
            mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=96, hop_length=1024, n_fft=2048)
            print(mel.shape)
            mel_db = librosa.power_to_db(mel)
            cqt = librosa.cqt(audio, sr=sr, n_bins=96, hop_length=1024, bins_per_octave=12)
            print(cqt.shape)
            cqt_mag = np.abs(cqt)

            # Save in individual folder
            seg_name = f"{track_name}_{segment_count}"
            seg_folder = os.path.join("Mix_features", seg_name)
            os.makedirs(seg_folder, exist_ok=True)

            sf.write(os.path.join(seg_folder, f"{seg_name}.wav"), audio, sr)
            np.save(os.path.join(seg_folder, f"{seg_name}_mel.npy"), mel_db)
            np.save(os.path.join(seg_folder, f"{seg_name}_cqt.npy"), cqt_mag)

            # Accumulate global lists
            all_mels.append(mel_db)
            all_cqts.append(cqt_mag)
            all_labels.append(label_vector)

            segment_count += 1

    # Save global lists
    np.save("Mix_features/all_mels.npy", np.array(all_mels))
    np.save("Mix_features/all_cqts.npy", np.array(all_cqts))
    np.save("Mix_features/all_labels.npy", np.array(all_labels))

## Segmentation and feature extraction

In [ ]:
track_names = [name for name in os.listdir(AUDIO_DIR) if os.path.isdir(os.path.join(AUDIO_DIR, name))]
extract_multilabel_segments(track_names, max_segments_per_track=None)


Procesando track: AClassicEducation_NightOwl
  → 12 intervalos de interés encontrados para AClassicEducation_NightOwl
    Intervalo 1: 5.15s – 8.31s  | Activos: ['clean_electric_guitar', 'drum_set', 'electric_bass', 'male_vocals', 'other_percussion', 'synthesizer']
    Intervalo 2: 24.24s – 27.59s  | Activos: ['clean_electric_guitar', 'drum_set', 'electric_bass', 'male_vocals', 'other_percussion', 'synthesizer']
    Intervalo 3: 29.44s – 32.69s  | Activos: ['clean_electric_guitar', 'drum_set', 'electric_bass', 'male_vocals', 'other_percussion', 'synthesizer']
    Intervalo 4: 32.97s – 37.29s  | Activos: ['clean_electric_guitar', 'drum_set', 'electric_bass', 'male_vocals', 'other_percussion', 'synthesizer']
    Intervalo 5: 42.63s – 47.00s  | Activos: ['clean_electric_guitar', 'drum_set', 'electric_bass', 'male_vocals', 'other_percussion', 'synthesizer']
    Intervalo 6: 48.58s – 51.87s  | Activos: ['clean_electric_guitar', 'drum_set', 'electric_bass', 'other_percussion', 'synthesizer'

Extrayendo segmentos de AClassicEducation_NightOwl:   0%|          | 0/24 [00:00<?, ?it/s]

[1. 1. 0. 1. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0.]


Extrayendo segmentos de AClassicEducation_NightOwl:  21%|██        | 5/24 [00:01<00:04,  4.46it/s]

(96, 130)
(96, 130)
[1. 1. 0. 1. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0.]
(96, 130)
(96, 130)
[1. 1. 0. 1. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0.]
(96, 130)
(96, 130)
[1. 1. 0. 1. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0.]
(96, 130)
(96, 130)
[1. 1. 0. 1. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0.]
(96, 130)
(96, 130)
[1. 1. 0. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
(96, 130)
(96, 130)
[1. 1. 0. 1. 1. 0. 0. 0. 0. 1. 1. 0. 0. 0. 1. 0. 0. 1.]


...

(96, 130)
[0. 0. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
(96, 130)
(96, 130)
[0. 0. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
(96, 130)
(96, 130)
[0. 0. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
(96, 130)


Extrayendo segmentos de Wolf_DieBekherte:  99%|█████████▉| 99/100 [00:10<00:00, 11.23it/s]

(96, 130)
[0. 0. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
(96, 130)
(96, 130)
[0. 0. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
(96, 130)
(96, 130)
[0. 0. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
(96, 130)


Extrayendo segmentos de Wolf_DieBekherte: 100%|██████████| 100/100 [00:10<00:00,  9.85it/s]


(96, 130)


Checking labels:

In [8]:
labels = np.load("Mix_features/all_labels.npy", allow_pickle=True)

In [9]:
labels[0]

array([1., 1., 0., 1., 1., 0., 0., 0., 0., 0., 1., 0., 0., 0., 1., 0., 0.,
       0.], dtype=float32)